# NavieBayes

In [1]:
import numpy as np
from collections import Counter, defaultdict

### 手搓NavieBayes

In [ ]:
class CategoricalNaiveBayes:
    def __init__(self, alpha = 1.0):
        self.alpha = alpha # 平滑系数
        self.classes_ = None # 所有的类别
        self.class_count_ = None # 每个类别的数量
        self.class_log_prior = None # 每个类别的先验概率
        self.featrue_values_ = None # 每个特征的可能取集合
        self.feature_count_ = None # 统计 P(X_j=value | Y=c) 的频数
        self.n_features_ = None # 特征数量

    def fit(self, X, y):
        X = np.asarray(X, dtype = object)
        y = np.asarray(y, dtype = object)

        if X.ndim != 2:
            raise ValueError("X必须是二维数组")
        if y.ndim != 1:
            raise ValueError("y必须是一维数组")
        if X.shape[0] != y.shape[0]:
            raise ValueError("X和y的样本数量不一样")

        n_samples, n_features = X.shape
        self.n_features_ = n_features
        self.classes_ = np.unique(y) # 有多少类别
        self.class_count_ = Counter(y) # 每个类别有多少数量,是一个字典
        self.feature_values_ = [] # 去统计每个特征的所有可能取值，是一个二维数组（特征，取值）
        for j in range(n_features):
            self.feature_values_.append(np.unique(X[:, j]))
        self.class_log_prior_ = {} #去统计每个类别的先验概率，是一个字典{类别：概率}
        K = len(self.classes_)
        for c in self.classes_:
            numerator = self.class_count_[c] + self.alpha
            denominator = n_samples + K * self.alpha
            if numerator == 0: # 如果为0，取负无穷，避免为0
                self.class_log_prior_[c] = -np.inf
            else:
                self.class_log_prior_[c] = np.log(numerator / denominator)
        self.feature_count_ = defaultdict(int) # 创建频数字典 {(y, j, xi[j]) : 数量}
        for xi, yi in zip(X, y):
            for j in range(n_features):
                self.feature_count_[(yi, j, xi[j])] += 1 # 计数
        return self

    def _feature_log_prob(self, c, j, value):
        """
        计算 log P(X_j=value | Y=c)
        """
        count = self.feature_count_[(c, j, value)] # type: ignore 取出该类型的数量
        class_count = self.class_count_[c] # type: ignore 计算y = c这个类别的数量
        S_j = len(self.feature_values_[j]) # 计算第j个特征可能取值的数量
        numerator = count + self.alpha
        denominator = class_count + S_j * self.alpha
        if numerator == 0: # 如果为0就直接取负无穷，避免为0
            return -np.inf
        return np.log(numerator / denominator)

    def predict_log_proba(self, X):
        """
        计算每个样本属于每个类别的 log 后验得分

        注意：
        这里输出的是未归一化的 log score：
            log P(Y=c) + Σ_j log P(X_j=x_j | Y=c)

        对分类来说，比较大小即可。
        """
        X = np.asarray(X, dtype = object)
        if X.ndim == 1:
            X = X.reshape(1, -1)

        if X.shape[1] != self.n_features_:
            raise ValueError("输入特征数量与训练时不一致")
        log_scores = []
        for xi in X:
            sample_scores = []
            for c in self.classes_: # type: ignore
                log_score = self.class_log_prior_[c]
                for j in range(self.n_features_):
                    log_score += self._feature_log_prob(c, j, xi[j]) # 每个特征的概率
                sample_scores.append(log_score)
            log_scores.append(sample_scores) # 数组的数组，对每个类别进行分类了
        return np.array(log_scores)

    def predict_proba(self, X):
        log_scores = self.predict_log_proba(X)
        # 使用 log-sum-exp 技巧做归一化，避免数值下溢
        max_log = np.max(log_scores, axis = 1, keepdims = True) # max是降维运算，keepdims是要保持维度
        exp_scores = np.exp(log_scores - max_log)
        proba = exp_scores / np.sum(exp_scores, axis = 1, keepdims = True)
        return proba

    def predict(self, X):
        log_scores = self.predict_log_proba(X)
        best_indices = np.argmax(log_scores, axis = 1)
        return self.classes_[best_indices]

#### 实现结果

In [15]:
X_train = np.array([
    [1, "S"],[1, "M"],[1, "M"],[1, "S"],
    [1, "S"],[2, "S"],[2, "M"],[2, "M"],
    [2, "L"],[2, "L"],[3, "L"],[3, "M"],
    [3, "M"],[3, "L"],[3, "L"],
], dtype=object)

y_train = np.array([
    -1, -1, 1, 1,
    -1, -1, -1, 1,
     1, 1, 1, 1,
     1, 1,-1,
], dtype=int)
X_test = np.array([[2, "S"]], dtype=object)
model_mle = CategoricalNaiveBayes(alpha=0.0)

model_mle.fit(X_train, y_train)

print("类别顺序：", model_mle.classes_)
print("log 得分：", model_mle.predict_log_proba(X_test))
print("后验概率：", model_mle.predict_proba(X_test))
print("预测类别：", model_mle.predict(X_test))

model_laplace = CategoricalNaiveBayes(alpha=1.0)

model_laplace.fit(X_train, y_train)

print("类别顺序：", model_laplace.classes_)
print("log 得分：", model_laplace.predict_log_proba(X_test))
print("后验概率：", model_laplace.predict_proba(X_test))
print("预测类别：", model_laplace.predict(X_test))

类别顺序： [-1 1]
log 得分： [[-2.7080502  -3.80666249]]
后验概率： [[0.75 0.25]]
预测类别： [-1]
类别顺序： [-1 1]
log 得分： [[-2.7968457  -3.42100001]]
后验概率： [[0.65116279 0.34883721]]
预测类别： [-1]


## sklearn.naive_bayes

In [16]:
from sklearn.naive_bayes import CategoricalNB # CategoricalNB 就是 sklearn 中用于类别型离散特征的朴素贝叶斯分类器
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder

model = make_pipeline(OrdinalEncoder(dtype = np.int64), CategoricalNB(alpha = 1.0))
print(y_train)
model.fit(X_train, y_train)
X_test = np.array([
    [2, "S"]
], dtype = object)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

print("预测类别：", y_pred)
print("类别顺序：", model.named_steps["categoricalnb"].classes_)
print("后验概率：", y_proba)

[-1 -1  1  1 -1 -1 -1  1  1  1  1  1  1  1 -1]
预测类别： [-1]
类别顺序： [-1  1]
后验概率： [[0.64 0.36]]
